# Payment Guard - extractor Qwen2.5 (LoRA, Unsloth)
Runtime > T4 GPU. Cambia SIZE en la celda 2. El modelo solo EXTRAE campos; nunca decide pagos.


In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
import os, getpass
HF_USER = "P1p2gamer26"
SIZE = "1.5B"  # "1.5B" (16-bit LoRA) o "3B" (QLoRA 4-bit); 0.5B ya probado
MODEL_ID = f"{HF_USER}/payment-guard-extractor-{SIZE.lower()}"
BASE = f"unsloth/Qwen2.5-{SIZE}-Instruct"
LOAD_4BIT = SIZE in ("3B", "7B")
BATCH, ACCUM = (2, 8) if LOAD_4BIT else (4, 4)
# Cambia por el raw del repo publico una vez publicado (o sube ml/train.jsonl con el panel de archivos de Colab)
TRAIN_URL = "https://raw.githubusercontent.com/P1p2gamer26/payment-guard/main/ml/train.jsonl"
if not os.path.exists("train.jsonl"):
    !wget -q -O train.jsonl $TRAIN_URL || echo "descarga fallida: sube train.jsonl a mano"
!wc -l train.jsonl

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=2048, load_in_4bit=LOAD_4BIT)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], random_state=30)
ds = load_dataset("json", data_files="train.jsonl", split="train")
ds = ds.map(lambda b: {"text": [tok.apply_chat_template(m, tokenize=False) for m in b["messages"]]}, batched=True)
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=SFTConfig(
    dataset_text_field="text", max_seq_length=2048, per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="linear", warmup_steps=8, logging_steps=5,
    fp16=True, optim="adamw_8bit", output_dir="out", report_to="none", seed=30))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

In [ ]:
# prueba rapida con una factura nueva
FastLanguageModel.for_inference(model)
import json
from unsloth import get_chat_template
msgs = json.loads(open("train.jsonl").readline())["messages"][:2]
x = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
print(tok.decode(model.generate(input_ids=x, max_new_tokens=200, do_sample=False)[0][x.shape[1]:], skip_special_tokens=True))

In [ ]:
# Token HF con permiso WRITE: se lee de Secretos (HF_TOKEN), NO se guarda en el cuaderno.
import os
from huggingface_hub import HfApi
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN").strip()
except Exception:
    HF_TOKEN = getpass.getpass("HF token (write): ").strip()
api = HfApi(token=HF_TOKEN)
print("Token OK para:", api.whoami()["name"])  # falla aqui en 1 s si el token no sirve

# 1) convertir y guardar en disco (no se pierde si la subida falla)
import glob
ggufs = lambda: [f for d in ("/content", "/tmp") for f in glob.glob(d + "/**/*.gguf", recursive=True) if "q4_k_m" in f.lower()]
if not ggufs():
    model.save_pretrained_gguf("gguf", tok, quantization_method="q4_k_m")
files = ggufs(); print(files)

# 2) subir
api.create_repo(MODEL_ID, exist_ok=True)
for f in files:
    api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=MODEL_ID)
print("https://huggingface.co/" + MODEL_ID)


In [ ]:
# Sube el Space (servidor llama.cpp del modelo) con el mismo token. Correr después de la celda anterior.
import urllib.request, os
os.makedirs("space", exist_ok=True)
for f in ("Dockerfile", "README.md", "start.py"):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/P1p2gamer26/payment-guard/main/space/{f}", f"space/{f}")
SPACE = f"{HF_USER}/payment-guard"
api.create_repo(SPACE, repo_type="space", space_sdk="docker", exist_ok=True)
api.add_space_variable(SPACE, "MODEL_REPO", MODEL_ID)
api.upload_folder(folder_path="space", repo_id=SPACE, repo_type="space")
print("https://huggingface.co/spaces/" + SPACE)
